# EDA 03: chọn ngưỡng lọc chuỗi outlier

**Mục đích:** chọn cách lọc chuỗi điểm GPS lỗi, theo hai cách và so sánh:
- **Cách N:** xoá chuỗi có số điểm `k ≤ N`.
- **Cách T:** xoá chuỗi có thời gian từ A đến B `dur_s ≤ T` (giây).

Notebook này không tự chọn ngưỡng. Nó tính các số liệu để bạn so sánh, và ghi tiêu chí chốt ở cuối.

**Chuỗi là gì.** Một đoạn gồm các điểm X lệch khỏi đường đi thật, nằm giữa hai điểm A và B đáng tin:

```
A ──► X1 ... Xk ──► B
```

Một chuỗi được ghi nhận khi cả bước vào (A→X1) và bước ra (Xk→B) đều nhanh hơn 300 km/h, khoảng cách thời gian mỗi bước không quá 30 phút, và cùng một file. Notebook đếm rộng (tối đa 50 điểm, A→B tối đa 2 giờ), rồi áp dụng N hoặc T lên tập này.

**Hai đại lượng:**
- `k`: số điểm X trong chuỗi.
- `dur_s`: thời gian từ A đến B, tính bằng giây.

**Điều kiện `back_ratio` đã bỏ khỏi phần quyết định.** Mọi bảng trong notebook này đếm trên toàn bộ chuỗi, để hai cách được so sánh trên cùng một tập.

**Cách chạy.** Chạy `python src/preprocess/convert_raw.py` một lần trước để tạo `data/interim`. Sau đó Run All, mất khoảng 1–2 phút. Muốn chạy thử thì đặt `USERS_LIMIT` ở cell đầu.

**Các bước:**
1. Tập chuỗi và phân bố cơ bản.
2. Lưới ứng viên N và T với các chỉ số cho từng ứng viên.
3. Tìm điểm gập: nơi số điểm bị xoá tăng đột ngột khi nới thêm một bậc.
4. So hai cách ở cùng số điểm bị xoá.
5. Bảng 2×2 cho cặp ngưỡng đã chọn.
6. Kiểm tra theo nhóm ghi dày và thưa.
7. Tiêu chí chốt (cuối notebook).


In [1]:
from pathlib import Path
import sys

import numpy as np
import pandas as pd

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "src").is_dir())
sys.path.insert(0, str(ROOT / "src"))
from eda.config import *          # dữ kiện từ tài liệu, mốc mô tả, bins (xem src/eda/config.py)
from eda.utils import fmt_s
from eda.scan import run
from eda import split

USERS_LIMIT = split.dev_users()   # nhóm dev (D17); đặt None để chạy mọi user (không dùng để chọn ngưỡng)

pd.set_option("display.max_columns", 30)

R = run(topics=["outliers"], users=USERS_LIMIT)
users, exc = R.users, R.exc
print(f"{len(users)} user, {len(exc):,} chuỗi tìm được")


[20/122] đã đọc tới user 031 (4 giây)
[40/122] đã đọc tới user 058 (7 giây)
[60/122] đã đọc tới user 091 (9 giây)
[80/122] đã đọc tới user 117 (10 giây)
[100/122] đã đọc tới user 147 (22 giây)
[120/122] đã đọc tới user 177 (24 giây)
[122/122] đã đọc tới user 179 (24 giây)
122 user, 327 chuỗi tìm được


## Bước 1: tập chuỗi

Thêm khoảng ghi điển hình của từng user vào mỗi chuỗi, để phân nhóm ghi dày và thưa ở bước 6. Ngưỡng 5 giây được chọn theo mục 4 (notebook 02): khoảng ghi điển hình của đa số user nằm ở 1–5 giây.


In [2]:
exc = R.exc.merge(users[["user", "median_dt_s"]], on="user", how="left")
GHI_THUA_S = 5                                  # khoảng ghi điển hình > 5 giây là ghi thưa
exc["ghi_thưa"] = exc["median_dt_s"] > GHI_THUA_S

COLS = ["user", "file", "start_utc", "lat_a", "lon_a", "k", "dur_s", "ab_m", "max_dist_m", "max_kmh", "airplane"]
print(f"Tập chuỗi: {len(exc):,} chuỗi, {int(exc['k'].sum()):,} điểm X, "
      f"{exc['user'].nunique()} user; trong nhãn máy bay: {int(exc['airplane'].sum())}")


Tập chuỗi: 327 chuỗi, 201,616 điểm X, 59 user; trong nhãn máy bay: 7


## Bước 2: lưới ứng viên và các chỉ số

Với mỗi ứng viên, bảng tính:
- **số điểm bị xoá**: tổng `k` của các chuỗi bị xoá. Đây là chỉ số chính.
- **điểm tăng thêm**: số điểm bị xoá thêm so với bậc trước. Dùng để tìm điểm gập ở bước 3.
- **số chuỗi, số user, k lớn nhất, thời lượng lớn nhất**: để xem chuỗi bị xoá có quá dài hay không.
- **chuỗi nhãn máy bay**: số chuỗi nằm trong đoạn có nhãn máy bay. Chỉ có dữ liệu cho các user có file nhãn, nên 0 không chứng minh được là không có chuyến bay bị xoá nhầm.

Lưới T gồm 10 giây đến 2 giờ. Lưới N gồm 1, 2, 3, 5, 10, 20, 50.


In [3]:
T_GRID = [10, 30, 60, 120, 300, 600, 1800, 3600, 7200]
N_GRID = [1, 2, 3, 5, 10, 20, 50]

def thong_ke(mask):
    s = exc[mask]
    return {"số_chuỗi": len(s),
            "số_điểm_bị_xoá": int(s["k"].sum()),
            "số_user": s["user"].nunique(),
            "thời_lượng_lớn_nhất": fmt_s(s["dur_s"].max()) if len(s) else "-",
            "k_lớn_nhất": int(s["k"].max()) if len(s) else 0,
            "chuỗi_nhãn_máy_bay": int(s["airplane"].sum())}

bang_T = pd.DataFrame([{"T": fmt_s(T), **thong_ke(exc["dur_s"] <= T)} for T in T_GRID]).set_index("T")
bang_T["điểm_tăng_thêm"] = bang_T["số_điểm_bị_xoá"].diff()

bang_N = pd.DataFrame([{"N": n, **thong_ke(exc["k"] <= n)} for n in N_GRID]).set_index("N")
bang_N["điểm_tăng_thêm"] = bang_N["số_điểm_bị_xoá"].diff()

print("Lọc theo T: xoá chuỗi có thời gian A→B ≤ T")
display(bang_T)
print("Lọc theo N: xoá chuỗi có số điểm k ≤ N")
display(bang_N)


Lọc theo T: xoá chuỗi có thời gian A→B ≤ T


,số_chuỗi,số_điểm_bị_xoá,số_user,thời_lượng_lớn_nhất,k_lớn_nhất,chuỗi_nhãn_máy_bay,điểm_tăng_thêm
T,,,,,,,
10 giây,10,31,4,7 giây,5,0,NaN
30 giây,14,58,6,30 giây,14,0,27.0
1 phút,24,205,11,1 phút,39,1,147.0
2 phút,46,719,18,1.91667 phút,105,2,514.0
5 phút,93,2377,29,4.91667 phút,144,4,1658.0
10 phút,136,6544,41,10 phút,389,5,4167.0
30 phút,201,18240,51,29.5333 phút,1149,6,11696.0
1 giờ,260,52448,56,59.8333 phút,3099,7,34208.0
2 giờ,297,92780,57,1.94583 giờ,5624,7,40332.0


Lọc theo N: xoá chuỗi có số điểm k ≤ N


,số_chuỗi,số_điểm_bị_xoá,số_user,thời_lượng_lớn_nhất,k_lớn_nhất,chuỗi_nhãn_máy_bay,điểm_tăng_thêm
N,,,,,,,
1,2,2,2,4 giây,1,0,NaN
2,8,14,5,19.7833 phút,2,0,12.0
3,18,44,9,19.7833 phút,3,1,30.0
5,32,109,15,19.7833 phút,5,2,65.0
10,48,227,21,29.5333 phút,10,2,118.0
20,69,513,27,29.5333 phút,18,3,286.0
50,115,2066,37,39.6333 phút,50,5,1553.0


## Bước 3: tìm điểm gập

**Điểm gập** là bậc mà số điểm bị xoá tăng gấp đôi trở lên so với bậc trước. Khi đó, nới ngưỡng thêm một bậc sẽ xoá nhiều điểm hơn hẳn, và rất có thể bắt đầu xoá các đoạn đi thật.

**Ứng viên** là bậc ngay trước điểm gập. Ứng viên của T và N được dùng làm `T_CHON` và `N_CHON` ở bước 5, không cần chọn tay. Hệ số `KNEE_RATIO = 2` là mốc mình chọn để mô tả; bạn có thể đổi và xem ứng viên thay đổi thế nào.


In [4]:
KNEE_RATIO = 2

def diem_gap(bang, grid):
    """Trả về bậc ngay trước điểm gập (giá trị số), hoặc None nếu không có điểm gập."""
    d = bang["điểm_tăng_thêm"]
    ratio = d / d.shift(1)
    labels = list(bang.index)
    hits = [i for i, lab in enumerate(labels) if d[lab] > 0 and ratio[lab] >= KNEE_RATIO]
    if not hits or hits[0] == 0:
        return None
    return grid[hits[0] - 1]

ung_vien_T = diem_gap(bang_T, T_GRID)
ung_vien_N = diem_gap(bang_N, N_GRID)
print(f"Ứng viên T: {fmt_s(ung_vien_T) if ung_vien_T else 'không có điểm gập'}")
print(f"Ứng viên N: {ung_vien_N if ung_vien_N else 'không có điểm gập'}")

# Hai ngưỡng này được dùng cho bước 5; không chọn tay.
T_CHON = ung_vien_T
N_CHON = ung_vien_N

Ứng viên T: 30 giây
Ứng viên N: 2


## Bước 4: so hai cách ở cùng số điểm bị xoá

Bước 3 cho mỗi cách một ứng viên, nhưng hai ứng viên có thể xoá số điểm rất khác nhau. Để so công bằng, bảng này lấy mỗi giá trị N, rồi tìm T xoá số điểm gần nhất với N đó.

Cột **chênh lệch** cho biết hai cách lệch nhau bao nhiêu điểm. Nếu chênh lệch nhỏ, hai cách tương đương về số lượng, và việc chọn phụ thuộc vào các chỉ số khác (thời lượng lớn nhất, nhãn máy bay).


In [5]:
s_sorted = exc.sort_values("dur_s")
cum_k = s_sorted["k"].cumsum().to_numpy()
dur_sorted = s_sorted["dur_s"].to_numpy()

def T_theo_so_diem(b):
    """T nhỏ nhất để số điểm bị xoá (lọc theo T) đạt ít nhất b."""
    i = min(np.searchsorted(cum_k, b), len(cum_k) - 1)
    return float(dur_sorted[i])

rows = []
for n in N_GRID:
    diem_N = thong_ke(exc["k"] <= n)["số_điểm_bị_xoá"]
    T_eq = T_theo_so_diem(diem_N)
    diem_T = thong_ke(exc["dur_s"] <= T_eq)["số_điểm_bị_xoá"]
    rows.append({"N": n, "điểm bị xoá (N)": diem_N, "T tương đương": fmt_s(T_eq),
                 "điểm bị xoá (T)": diem_T, "chênh lệch": diem_T - diem_N})
display(pd.DataFrame(rows).set_index("N"))


,điểm bị xoá (N),T tương đương,điểm bị xoá (T),chênh lệch
N,,,,
1,2,4 giây,8,6
2,14,6 giây,29,15
3,44,15 giây,45,1
5,109,52 giây,118,9
10,227,1.05 phút,236,9
20,513,1.58333 phút,529,16
50,2066,4.31667 phút,2067,1


## Bước 5: bảng 2×2 cho cặp ngưỡng đã chọn

`N_CHON` và `T_CHON` được lấy tự động từ ứng viên ở bước 3, nên không cần đặt tay. Bảng cho biết mỗi chuỗi nằm ở đâu:
- **cả hai bắt:** cả N và T đều xoá chuỗi này.
- **chỉ N bắt:** chuỗi có `k ≤ N` nhưng dài hơn T. Đây là các chuỗi lâu, cần xem thêm.
- **chỉ T bắt:** chuỗi ngắn theo thời gian nhưng có nhiều hơn N điểm.
- **cả hai không bắt:** giữ lại.

Hai bảng dưới liệt kê các chuỗi thuộc hai nhóm "chỉ một cách bắt". Đây là nơi hai cách khác nhau rõ nhất; so các cột `max_kmh` và `max_dist_m` của hai nhóm để thấy khác biệt.


In [6]:
if N_CHON is None or T_CHON is None:
    print("Chưa có ứng viên cho cả hai cách, bỏ qua bước 5.")
    bat_N = pd.Series(False, index=exc.index)
    bat_T = pd.Series(False, index=exc.index)
else:
    bat_N = exc["k"] <= N_CHON
    bat_T = exc["dur_s"] <= T_CHON

nhan_T = fmt_s(T_CHON) if T_CHON else "-"
nhom = {
    f"cả hai bắt (k ≤ {N_CHON} và A→B ≤ {nhan_T})": bat_N & bat_T,
    f"chỉ N bắt (k ≤ {N_CHON}, A→B > {nhan_T})": bat_N & ~bat_T,
    f"chỉ T bắt (A→B ≤ {nhan_T}, k > {N_CHON})": ~bat_N & bat_T,
    "cả hai không bắt": ~bat_N & ~bat_T,
}
bang_22 = pd.DataFrame({
    "số_chuỗi": [int(m.sum()) for m in nhom.values()],
    "số_điểm": [int(exc.loc[m, "k"].sum()) for m in nhom.values()],
}, index=list(nhom.keys()))
display(bang_22)

print("10 chuỗi chỉ N bắt (dài nhất):")
display(exc[bat_N & ~bat_T].nlargest(10, "dur_s")[COLS].round(1))
print("10 chuỗi chỉ T bắt (nhiều điểm nhất):")
display(exc[bat_T & ~bat_N].nlargest(10, "k")[COLS].round(1))

,số_chuỗi,số_điểm
cả hai bắt (k ≤ 2 và A→B ≤ 30 giây),5,8
"chỉ N bắt (k ≤ 2, A→B > 30 giây)",3,6
"chỉ T bắt (A→B ≤ 30 giây, k > 2)",9,50
cả hai không bắt,310,201552


10 chuỗi chỉ N bắt (dài nhất):


,user,file,start_utc,lat_a,lon_a,k,dur_s,ab_m,max_dist_m,max_kmh,airplane
121,042,20090317033057.plt,2009-03-17 06:01:12+00:00,39.7,116.3,2,1187.0,177.1,2711.5,1839.6,False
278,144,20081105011825.plt,2008-11-05 01:25:15+00:00,40.1,116.6,2,254.0,20031.3,13376.7,512.7,False
244,128,20100124123200.plt,2010-01-24 13:02:16+00:00,39.8,116.3,2,48.0,260.7,849.0,1528.2,False


10 chuỗi chỉ T bắt (nhiều điểm nhất):


,user,file,start_utc,lat_a,lon_a,k,dur_s,ab_m,max_dist_m,max_kmh,airplane
288,144,20090321083221.plt,2009-03-21 08:40:51+00:00,1.3,104.0,14,15.0,1249.8,1166.1,351.1,False
181,082,20080801051036.plt,2008-08-01 08:44:06+00:00,40.2,116.6,8,22.0,1781.7,1597.4,351.0,False
63,018,20081205005116.plt,2008-12-05 14:31:00+00:00,40.0,116.1,5,6.0,482.2,397.3,306.9,False
300,153,20091129234104.plt,2009-11-30 02:30:39+00:00,39.9,116.4,5,6.0,82.7,131.5,473.5,False
303,153,20091129234104.plt,2009-11-30 02:30:56+00:00,39.9,116.4,5,6.0,10.5,122.0,439.2,False
305,153,20091129234104.plt,2009-11-30 02:31:06+00:00,39.9,116.4,4,5.0,263.0,198.3,407.7,False
284,144,20081218030846.plt,2008-12-18 03:11:40+00:00,31.2,121.3,3,20.0,1577.7,1115.8,356.8,False
302,153,20091129234104.plt,2009-11-30 02:30:52+00:00,39.9,116.4,3,4.0,248.0,199.3,414.5,False
304,153,20091129234104.plt,2009-11-30 02:31:02+00:00,39.9,116.4,3,4.0,106.8,92.0,326.7,False


## Bước 6: kiểm tra theo nhóm ghi dày và thưa

Mỗi bảng đếm số chuỗi bị xoá cho từng nhóm user. Nếu một cách cho kết quả ổn định ở cả hai nhóm, cách đó đáng tin hơn. Nếu nhóm thưa gần như không bị ảnh hưởng ở mọi ngưỡng, ghi đây là điểm chưa giải quyết, vì khoảng ghi thưa làm chuỗi khó xuất hiện đúng dạng quy tắc đang kiểm tra.


In [7]:
def dem_theo_nhom(cot, gia_tri, nhan):
    ket_qua = {}
    for thua, d in exc.groupby("ghi_thưa"):
        ket_qua["ghi thưa (> 5 s)" if thua else "ghi dày (≤ 5 s)"] = [int((d[cot] <= g).sum()) for g in gia_tri]
    return pd.DataFrame(ket_qua, index=nhan).T

print("Số chuỗi bị xoá, lọc theo T:")
display(dem_theo_nhom("dur_s", T_GRID, [fmt_s(T) for T in T_GRID]))
print("Số chuỗi bị xoá, lọc theo N:")
display(dem_theo_nhom("k", N_GRID, N_GRID))


Số chuỗi bị xoá, lọc theo T:


,10 giây,30 giây,1 phút,2 phút,5 phút,10 phút,30 phút,1 giờ,2 giờ
ghi dày (≤ 5 s),10,14,24,45,89,129,193,250,286
ghi thưa (> 5 s),0,0,0,1,4,7,8,10,11


Số chuỗi bị xoá, lọc theo N:


,1,2,3,5,10,20,50
ghi dày (≤ 5 s),2,8,16,27,41,61,106
ghi thưa (> 5 s),0,0,2,5,7,8,9


## Tiêu chí chốt ngưỡng

Ghi trước khi xem kết quả cuối cùng. Các tiêu chí dưới đây là đề xuất, bạn có thể sửa.

1. **Cách và giá trị:** chọn cách (N hoặc T) có điểm gập rõ, rồi lấy ứng viên ngay trước điểm gập (bước 3).
2. **Cùng số điểm:** nếu hai cách cho số điểm bị xoá gần nhau (bước 4), chọn cách có thời lượng lớn nhất của chuỗi bị xoá thấp hơn.
3. **Máy bay:** không có chuỗi nhãn máy bay bị xoá trong các user có nhãn (bước 2 và bước 5).
4. **Nhóm ghi:** cả nhóm dày và thưa có phản ứng tương tự (bước 6). Nếu không, ghi rõ là chưa giải quyết.

**Quyết định:**
- Cách: **N** (xoá theo số điểm của chuỗi), N thuần.
- Giá trị: **N = 5**, tức xoá chuỗi có `k ≤ 5`. Khi chạy pipeline: xoá 455 điểm, đúng như bảng ở bước 2.
- Lý do:
  - Ở cùng số điểm bị xoá, hai cách chỉ lệch nhau 3–20 điểm (bước 4), nên chọn theo việc mỗi cách xoá chuỗi nào.
  - T phụ thuộc tần suất ghi: với T = 30 giây, nhóm ghi thưa bị xoá 0 chuỗi. N = 5 bắt chuỗi ở cả hai nhóm (bước 6).
  - Chuỗi chỉ N bắt có dấu hiệu lỗi rõ (`max_kmh` hàng nghìn đến hàng trăm nghìn km/h, X cách A tới 621 km). Chuỗi chỉ T bắt giống di chuyển nhanh thật (8–14 điểm liên tiếp, khoảng 350–410 km/h, lệch khoảng 1 km) (bước 5).
  - N = 5 là ứng viên ngay trước điểm gập (bước 3). Tính theo mỗi đơn vị N, số điểm tăng thêm gần như phẳng từ 3 đến 10, nên 5 không sát điểm gập.
  - Không thêm chốt thời gian: điều kiện A→B ≤ 30 phút chỉ giữ lại 2/359 chuỗi và làm quy tắc trông như kết hợp N với T, nên đã bỏ.
- Tiêu chí chưa đạt: tiêu chí 3, vì N = 5 xoá 2 chuỗi trong đoạn có nhãn máy bay. Đã ghi là hạn chế đã biết.
- Chi tiết: `notes/assumptions.md` (A6), `notes/decisions.md` (D5). Code: `remove_outliers` trong `src/preprocess/clean_trajectory.py`.
